# Minimum Set Cover: giới thiệu, chạy thuật toán và đánh giá

Notebook này cung cấp một interface tương tác để:

1. Giới thiệu bài toán Minimum Set Cover và các thuật toán trong project.
2. Sinh một dataset hợp lệ, có thể tái lập bằng `seed`.
3. Chạy lần lượt Greedy, Brute Force, Dynamic Programming và Branch and Bound.
4. So sánh số subset được chọn, nghiệm tối ưu và thời gian chạy.
5. Trực quan hóa kết quả bằng biểu đồ.

Chạy notebook từ thư mục gốc repository. Nếu chưa cài các thư viện notebook, chạy:

```bash
python -m pip install -e ".[notebook]"
```

## 1. Chuẩn bị môi trường

Các import dưới đây sử dụng trực tiếp source code trong thư mục `src/`.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from data_generator import generate_instance
from min_cover_set import MinimumCoverSet

## 2. Sinh dữ liệu

Các tham số có thể thay đổi trực tiếp để thử những kích thước/density khác nhau. Dataset được sinh bởi `generate_instance`, nên mọi phần tử trong universe luôn thuộc ít nhất một subset.

In [ ]:
NUM_ELEMENTS = 20
NUM_SUBSETS = 12
DENSITY = 0.30
SEED = 42

instance = generate_instance(
    num_elements=NUM_ELEMENTS,
    num_subsets=NUM_SUBSETS,
    density=DENSITY,
    seed=SEED,
)

subset_sizes = {
    str(name): len(elements)
    for name, elements in instance.subsets.items()
}

print(f"Universe: {len(instance.universe)} elements")
print(f"Subsets: {len(instance.subsets)}")
print(f"Density: {DENSITY}, seed: {SEED}")
pd.DataFrame(
    sorted(subset_sizes.items()),
    columns=["subset", "number_of_elements"],
)

## 3. Chạy lần lượt các thuật toán

Brute Force là nghiệm tham chiếu tối ưu. Với mỗi thuật toán, notebook đo riêng thời gian gọi hàm bằng `time.perf_counter()`.

In [ ]:
cover = MinimumCoverSet(instance)
algorithms = {
    "Greedy": cover.greedy,
    "Brute Force": cover.brute_force,
    "Dynamic Programming": cover.dynamic_programming,
    "Branch and Bound": cover.branch_and_bound,
}

results = []
optimal_solution = None
optimal_size = None

for name, algorithm in algorithms.items():
    start = time.perf_counter()
    selected = algorithm()
    elapsed_ms = (time.perf_counter() - start) * 1000

    if name == "Brute Force":
        optimal_solution = selected
        optimal_size = len(selected)

    results.append(
        {
            "algorithm": name,
            "selected": len(selected),
            "runtime_ms": elapsed_ms,
            "solution": list(selected),
        }
    )

results_df = pd.DataFrame(results)
results_df["optimal"] = results_df["selected"] == optimal_size
results_df["accuracy"] = optimal_size / results_df["selected"]
results_df[["algorithm", "selected", "optimal", "accuracy", "runtime_ms"]]

## 4. Kết quả và so sánh

- `selected`: số subset được thuật toán chọn.
- `optimal`: thuật toán có đạt số lượng tối thiểu theo Brute Force hay không.
- `accuracy`: tỷ lệ `optimal_size / selected`.
- `runtime_ms`: thời gian chạy thuật toán, tính bằng milliseconds.

Thời gian có thể dao động giữa các lần chạy; nên dùng cùng một dataset khi so sánh.

In [ ]:
display(
    results_df.assign(
        accuracy=results_df["accuracy"].round(2),
        runtime_ms=results_df["runtime_ms"].round(3),
    )
)

fastest = results_df.loc[results_df["runtime_ms"].idxmin()]
print(
    f"Fastest algorithm: {fastest['algorithm']} "
    f"({fastest['runtime_ms']:.3f} ms)"
)
print(f"Optimal solution size: {optimal_size}")

## 5. Trực quan hóa

Biểu đồ trái so sánh số subset được chọn; biểu đồ phải so sánh thời gian chạy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_df.plot.bar(
    x="algorithm",
    y="selected",
    ax=axes[0],
    color="#4C78A8",
    legend=False,
    title="Solution size",
)
axes[0].set_xlabel("")
axes[0].set_ylabel("Number of selected subsets")
axes[0].tick_params(axis="x", rotation=25)

results_df.plot.bar(
    x="algorithm",
    y="runtime_ms",
    ax=axes[1],
    color="#F58518",
    legend=False,
    title="Runtime comparison",
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Runtime (ms)")
axes[1].tick_params(axis="x", rotation=25)

fig.suptitle("Minimum Set Cover evaluation", fontsize=14)
fig.tight_layout()
plt.show()